# A1 2024.2 - Resolução

> **Atenção:** as questões que solicitarem o projeto de um algoritmo deverão conter em sua resposta (1) o pseudocódigo do algoritmo, ou (2) uma descrição textual sem ambiguidade com os passos necessários para implementá-lo.

---
### Questão 01 (2 pontos)

Determine uma função $f(n)$ para o código a seguir tal que $T(n)=\Theta(f(n))$. Explique como a solução foi encontrada.

```c
HashTable t;

void f(int v[], int startInx, int endInx) {
    if (startInx < endInx - 1) {
        int midInx = (startInx + endInx) / 2;
        f(v, startInx, midInx);
        f(v, midInx, endInx);

        for (int i=startInx; i < endInx; i++) {
            t[v[i]]++;
        }

        int r[endInx - startInx];
        int aInx = startInx;
        int bInx = midInx;
        int rInx = 0;

        while (aInx < midInx && bInx < endInx) {
            r[rInx] = v[aInx] <= v[bInx] ? v[aInx++] : v[bInx++];
            r[rInx++] += t[r[rInx]];
        }

        while (aInx < midInx) {
            r[rInx] = v[aInx++];
            r[rInx++] += t[r[rInx]];
        }

        while (bInx < endInx) {
            r[rInx] = v[bInx++];
            r[rInx++] += t[r[rInx]];
        }

        for (aInx = startInx; aInx < endInx; ++aInx) {
            v[aInx] = r[aInx - startInx];
        }
    }
}
```

**Resposta:**

A função `f` define um caso base `n<=1`, onde $T(1)$ é $\Theta(1)$

Em outros casos, a função `f` realiza os seguintes procedimentos:
- Calculo do ponto médio, em tempo constante;
- Duas chamadas recursivas de tamanho $\frac{n}{2}$;
- Definição de variáveis, em tempo constante;
- Um loop com dois ponteiros, um em cada lado, aproximando-os do ponto médio e dividindo em dois casos dependendo de quem chega primeiro:
    - Caso seja percorrida a metade da esquerda primeiro, completa-se a metade da direita após;
    - Caso seja percorrida a metade da direita primeiro, completa-se a metade da esquerda após;
    - Percorrendo todos os $n$ elementos ao final
- Finaliza iterando pelos $n$ elementos atualizando o vetor original.

Logo, ocusto da função `f`, para uma entrada de tamanho $n$, será:
$$
T(n) = 2T(\frac{n}{2})+2n+1=2T(\frac{n}{2}) + \Theta(n)
$$

Considerando a forma padrão do Teorema Mestre, $T(n) = aT(\frac{n}{b})+f(n)$, temos que $a=2$, $b=2$ e $f(n) = \Theta(n)=\Theta(n^{\log_b{a}}\log^k{n})$, para $k=0$. Logo, essa recursão se enquadra no caso $2$ do Teorema Mestre, de forma que $T(n)=\Theta(n^{\log_b{a}}\log^{k+1}{n})=\Theta(n\log{n})$.

---
### Questão 02 (1 ponto)

A seguir são apresentados cenários que exigem a escolha de um algoritmo de ordenação. Dentre os algoritmos apresentados em aula, proponha um adequado para cada cenário, buscando a maior eficiência de tempo e espaço. Explique a sua escolha.

**a)** Uma estação meteorológica precisa manter continuamente ordenados os $n$ maiores valores de temperatura em um sistema com memória limitada. Os dados chegam em tempo real, incluindo temperaturas com muitas casas decimais em uma ampla faixa de valores, e o conjunto é atualizado frequentemente. É essencial que o algoritmo apresente complexidade $O(n\log n)$ no pior cenário.

**b)** Uma empresa de logística precisa ordenar 10 milhões de códigos de rastreio de encomendas. Cada código consiste em 12 dígitos numéricos (por exemplo, `202407260134`). A empresa precisa ordenar esses códigos frequentemente para otimizar as rotas de entrega e o processo de triagem nos centros de distribuição. O sistema tem memória suficiente para processar todos os códigos de uma vez.

**a)** Para ordenar os dados em $O(n\log{n})$ existem, essencialmente, três opções: Merge Sort, Quick Sort, e Heap Sort. O algoritmo de Merge Sort exige um vetor auxiliar de tamanho $n$, de forma que não seja adequade devido à memória limitada no sistema. O algoritmo de Quick Sort, não exige memória adicional, entretanto, caso se tenha na escolha dos pivôs, possui pior caso em $O(n^2)$, não cumprindo o requisito de pior cenário. Por fim, o algoritmo de Heap Sort, também funciona in-place, e garante pior caso $O(n\log{n})$, pois as operações `heapfy_up` e `heapfy_down` possuem custo $O(\log{n})$ independente das entradas. Por tanto, o algoritmo mais adequado para esse cenário é o Heap Sort.

**b)** Dado que todos os códigos possuem exatamente o mesmo tamanho e pode ser atribuída uma ordem dígito a dígito, e o sistema possui memória para processar todos os códigos de uma única vez, pode-se implementar o Radix Sort, garantindo ordenação em $O(n)$.

---
### Questão 03 (2 pontos)

Uma empresa de análise de redes sociais, que possui uma base de dados com milhões de usuários, está desenvolvendo um algoritmo para identificar influenciadores emergentes. Além de informações cadastrais, em cada usuário é armazenado o seu número de seguidores e uma pontuação representando o seu índice de engajamento, calculado com base em uma fórmula envolvendo likes, comentários e compartilhamentos. Um influenciador emergente é um usuário que está no top 10% em termos de engajamento, porém não está no top 10% em número de seguidores. Projete um algoritmo eficiente que, dada uma sequência $A$ com $n$ tuplas $<id\_usuário, seguidores, engajamento>$, identifique todos os influenciadores emergentes. O algoritmo deverá apresentar uma complexidade $O(n)$.

*Caso seja necessário, é permitido nessa questão fazer referência a algoritmos apresentados em aula para compor a solução.*

In [17]:
def median_brute_force(v, low, high):
    if (high < low): return

    for i in range(low, high):
        for j in range(i, high):
            if (v[j] > v[j+1]):
                v[j], v[j+1] = v[j+1], v[j]

    return v[low + (high-low)//2]

def median_of_medians(v, low, high):
    if (high < low): return

    i = 0
    medians = list()
    pos = low
    while pos < high:
        resto = high-pos
        m = median_brute_force(v, pos, pos+4 if (resto>4) else pos+resto)
        medians.append(m)
        i+=1
        pos+=5

    return medians[0] if (i == 1) else median_of_medians(medians, 0, i-1)

def partition(v, low, high, pivot):
    if (high < low): return -1

    pivot_pos = -1
    for i in range(low, high+1):
        if (v[i] == pivot):
            v[i], v[high] = v[high], v[i]
            pivot_pos = high
            break
    if (pivot_pos == -1): return -1

    i = low
    while (i < high):
        if (v[i] < pivot):
            v[i], v[low] = v[low], v[i]
            low+=1
        i+=1
    # x >= pivot, para todo x em v[low:high] 
    v[low], v[pivot_pos] = v[pivot_pos], v[low]
    return low

def quick_select(v, low, high, k):
    if (high < low): return -1

    mom = median_of_medians(v, low, high)
    mom_pos = partition(v, low, high, mom)

    if (mom_pos == k-1):
        return mom_pos
    if (mom_pos > k-1):
        return quick_select(v, low, mom_pos-1, k)
    else:
        return quick_select(v, mom_pos+1, high, k)

def question_03(users:list)->list:
    n = len(users)

    seguidores = [u[1] for u in users]
    engajamentos = [u[2] for u in users]

    max_seg_pos = quick_select(seguidores, 0, n-1, (9*n)//10)
    min_eng_pos = quick_select(engajamentos, 0, n-1, (9*n)//10)

    max_seg = seguidores[max_seg_pos]
    min_eng = engajamentos[min_eng_pos]

    return [u for u in users if (u[1]<max_seg and u[2]>min_eng)]

---
### Questão 04 (5 pontos)

Uma sequência $A$ de tamanho $n$ contém inteiros positivos e negativos.

**a)** Projete um algoritmo $O(n^3)$ capaz de determinar os índices $i$ e $j$, sendo $i<j$ tal que a soma de $A[i]+...+A[j]$ é máxima.

**b)** Otimize a solução do item anterior produzindo um algoritmo $O(n^2)$.

**c)** Avalie se é possível produzir um algoritmo $O(n)$ para esse problema. Caso seja possível, apresente o algoritmo, caso contrário prove que é impossível.

*Nessa questão não é permitido apenas fazer referência a algoritmos apresentados em aula para compor a solução. Tais algoritmos podem ser utilizados, no entanto devem ser descritos explicitamente.*

In [ ]:
def question_04_a(v):
    # Note que o algoritmo abaixo é Theta(n^2), logo, também é O(n^3)

    n = len(v)
    if (n<=2): return None # impossível ter i < j

    max_sum = v[0] + v[1]
    best_low = 0
    best_high = 1
    for i in range(n):
        soma = v[i]
        for j in range(i+1, n):
            soma += v[j]
            if (soma > max_sum):
                max_sum = soma
                best_low = i
                best_high = j

    return best_low, best_high

def question_04_b(v):
    # Como dito em (a), o algoritmo é Theta(n^2)
    return question_04_a(v)

def question_04_c(v):
    # tentativa de adaptação de kadane para a exigência de low != high
    n = len(v)
    if (n<=2): return None # impossível ter low != high

    max_sum = v[0] + v[1]
    best_low = 0
    best_high = 1
    max_atual = max_sum
    best_low_atual = 0
    best_high_atual = 1
    i = 2
    while i < n:
        if (max_atual + v[i] > v[i]):
            best_high_atual = i
            max_atual += v[i]
            i += 1

            if (max_atual > max_sum):
                max_sum = max_atual
                best_high = best_high_atual
                best_low = best_low_atual

        elif (i<n-1):
            if (max_atual + v[i] + v[i+1] > v[i] + v[i+1]):
                best_high_atual = i+1
                max_atual += v[i] + v[i+1]
            else:
                best_low_atual = i
                best_high_atual = i+1
                max_atual = v[i] + v[i+1]
            
            i+=2
            if (max_atual > max_sum):
                max_sum = max_atual
                best_low = best_low_atual
                best_high = best_high_atual

        else:
            break # não tem mais duas entradas para um intervalo válido (low != high)

    return best_low, best_high

# print(question_04_a([1, 2, 3, 4, 5]))
# print(question_04_c([1, 2, 3, 4, 5]))
# print()
# print(question_04_a([1, 20, -3, 4, -5]))
# print(question_04_c([1, 20, -3, 4, -5]))

(0, 4)
(0, 4)

(0, 3)
(0, 3)
